# Modelo de Classificação: Regressão Logística como Baseline Linear

Este notebook documenta a implementação do algoritmo de Regressão Logística para a classificação do potencial de popularidade de obras literárias. Por se tratar de um modelo linear probabilístico, ele atuará como a nossa linha de base (*baseline*) metodológica.

Diferente de modelos de alta complexidade estrutural, a Regressão Logística nos oferece total transparência analítica através do isolamento dos coeficientes de cada variável, permitindo quantificar o impacto exato de cada metadado de pré-lançamento no sucesso comercial da obra.

In [2]:
%pip install pandas scikit-learn imbalanced-learn pyarrow joblib

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.under_sampling import RandomUnderSampler

# 1. Carga dos dados e cruzamento estrutural
df_generos = pd.read_parquet('../data/processed/books_pivot_mapped.parquet')
df_original = pd.read_parquet('../data/processed/books_clean.parquet')

df_class = pd.merge(
    df_original[['title', 'author', 'bookformat']], 
    df_generos, 
    on=['title', 'author'], 
    how='inner'
)

# 2. Definição do Alvo (3 Classes de Popularidade)
def definir_tres_classes(ratings):
    if ratings >= 10000:
        return 1  # Bestseller
    elif ratings >= 1000:
        return 2  # Média Popularidade
    else:
        return 3  # Nicho

df_class['popularity_class'] = df_class['totalratings'].apply(definir_tres_classes)

# 3. Engenharia de Atributos (Frequência do Autor e Dummies de Formato)
autor_frequencia = df_class['author'].value_counts()
df_class['author_frequency'] = df_class['author'].map(autor_frequencia)

top_formatos = df_class['bookformat'].value_counts().index[:5]
df_class['format_grouped'] = df_class['bookformat'].apply(lambda x: x if x in top_formatos else 'Other')
df_formatos_encoded = pd.get_dummies(df_class['format_grouped'], prefix='format', drop_first=True)

macro_generos = [
    'Artes, Lazer e Estilo de Vida', 'Fantasia e Ficção Científica', 
    'Ficção Geral e Literatura', 'História e Biografia', \
    'Infantojuvenil e Quadrinhos', 'Mistério, Thriller e Terror', \
    'Não-Ficção e Autodesenvolvimento', 'Outros', 'Romance'
]

# 4. Consolidação dos Preditores (X) e Alvo (y)
x = pd.concat([df_class[['pages', 'author_frequency']], df_formatos_encoded, df_class[macro_generos]], axis=1).astype(float)
y = df_class['popularity_class']

# 5. Divisão Hold-Out (70/30) Estratificada
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42, stratify=y)

# 6. Padronização Obrigatória (Z-Score)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 7. Balanceamento via Undersampling Físico para a Arena Justa
rus = RandomUnderSampler(random_state=42)
X_train_resampled, y_train_resampled = rus.fit_resample(X_train_scaled, y_train)

print(f"Dados prontos! Formato do treino balanceado linear: {X_train_resampled.shape}")

Dados prontos! Formato do treino balanceado linear: (7746, 16)


## Otimização e treino da Regressão Logística

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

# 1. Definição do espaço de busca para o hiperparâmetro C
param_grid_lr = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty': ['l2'],                  # Penalização Ridge para controlar multicolinearidade
    'solver': ['lbfgs'],
    'max_iter': [1000]                  # Teto estendido para garantir a convergência matemática
}

# 2. Configuração do GridSearch com Validação Cruzada (5 folds)
# multi_class='multinomial' garante a abordagem Softmax para as 3 classes simultâneas
grid_lr = GridSearchCV(
    estimator=LogisticRegression(random_state=42),
    param_grid=param_grid_lr,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

print("Iniciando a busca científica pelos melhores parâmetros da Regressão Logística...")
grid_lr.fit(X_train_resampled, y_train_resampled)

# 3. Extração do melhor modelo encontrado
melhor_lr = grid_lr.best_estimator_
print(f"Melhor parâmetro 'C' encontrado: {grid_lr.best_params_['C']}\n")

# 4. Avaliação final no conjunto de teste intacto
previsoes_lr = melhor_lr.predict(X_test_scaled)

print("=======================================================================")
print("          RELATÓRIO DE DESEMPENHO - REGRESSÃO LOGÍSTICA                ")
print("=======================================================================")
print(f"Acurácia Global: {accuracy_score(y_test, previsoes_lr) * 100:.2f}%\n")
print(classification_report(y_test, previsoes_lr, target_names=['Classe 1', 'Classe 2', 'Classe 3']))
print("=======================================================================")

Iniciando a busca científica pelos melhores parâmetros da Regressão Logística...
Melhor parâmetro 'C' encontrado: 0.1

          RELATÓRIO DE DESEMPENHO - REGRESSÃO LOGÍSTICA                
Acurácia Global: 60.60%

              precision    recall  f1-score   support

    Classe 1       0.14      0.62      0.22      1106
    Classe 2       0.32      0.39      0.35      4995
    Classe 3       0.91      0.66      0.77     18501

    accuracy                           0.61     24602
   macro avg       0.46      0.56      0.45     24602
weighted avg       0.76      0.61      0.66     24602



c:\Users\User3\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


## Extração de Coeficientes

In [6]:
# Criando um DataFrame com a importância linear dos atributos para a Classe 1 (Bestsellers)
coeficientes_classe1 = melhor_lr.coef_[0] # O índice 0 mapeia a primeira classe (Popularidade Alta)

df_coeficientes = pd.DataFrame({
    'Atributo': x.columns,
    'Coeficiente (Peso)': coeficientes_classe1
})

# Ordenando do impacto mais positivo para o mais negativo
df_coeficientes = df_coeficientes.sort_values(by='Coeficiente (Peso)', ascending=False).reset_index(drop=True)

print("=======================================================================")
# Coeficientes positivos aumentam a chance de ser Bestseller; negativos reduzem.
print("   IMPACTO DOS METADADOS NA PROBABILIDADE DE UM LIVRO SER BESTSELLER   ")
print("=======================================================================")
print(df_coeficientes.to_string(index=False, formatters={'Coeficiente (Peso)': '{:,.4f}'.format}))
print("=======================================================================")

   IMPACTO DOS METADADOS NA PROBABILIDADE DE UM LIVRO SER BESTSELLER   
                        Atributo Coeficiente (Peso)
       Ficção Geral e Literatura             0.4116
                           pages             0.3759
     Infantojuvenil e Quadrinhos             0.1590
     Mistério, Thriller e Terror             0.1306
                         Romance             0.1177
                author_frequency             0.1067
                          Outros             0.0915
    Fantasia e Ficção Científica             0.0899
            História e Biografia             0.0091
   Artes, Lazer e Estilo de Vida            -0.0031
    format_Mass Market Paperback            -0.0125
                    format_Other            -0.0151
Não-Ficção e Autodesenvolvimento            -0.0195
           format_Kindle Edition            -0.0290
                    format_ebook            -0.1375
                format_Paperback            -0.1537
